# QRR-BnB 与 GW-BnB 同问题复现对照

这本 notebook 补齐 Q-RBnBR S1 方法对照：QRR-BnB 使用 p=1 QAOA correlation，GW-BnB 使用 classical SDP matrix。两条路线共享 parity tree、变量消元、admissible bound、R1/R2/R3 与 exact leaf closure。

> 对照重点是 relaxation provider 如何影响搜索路径；两条 BnB 的 `optimal` 都来自完整 tree certificate，不来自 QRR 或 GW candidate 本身。

## 1. 环境与导入

In [1]:
import json
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_qubo_result
from lib.solvers.qubo import (
    ExactQuboSolver,
    GwBranchAndBoundQuboSolver,
    QrbnbrQuboSolver,
)
from problem.reproductions import build_qrbnbr_s1_instance
from tests.oracles import enumerate_qubo, public_json_number

print("Python:", sys.executable)

Python: c:\Users\peter\Documents\TaiyiQSolution\.venv\Scripts\python.exe


## 2. 共享问题与独立 oracle

In [2]:
problem = build_qrbnbr_s1_instance(
    variable_count=10,
    edge_probability=0.45,
    seed=2025,
)
oracle_rows = enumerate_qubo(problem)
oracle_energy = public_json_number(oracle_rows[0]["energy_exact"])
exact_result = ExactQuboSolver().solve(problem)

assert exact_result["best_energy"] == oracle_energy
print("Problem:", problem["problem_id"])
print("Oracle optimum:", oracle_energy)

Problem: qrbnbr-s1-er-n10-p0p45-seed2025
Oracle optimum: -14


## 3. R1：最大绝对 relaxation entry

两条路线使用同一 exact threshold、BFS traversal 和 R1，只替换 relaxation provider。

In [ ]:
qrr_common = {
    "brute_force_threshold": 4,
    "qaoa_grid_size": 5,
    "qaoa_refinement_steps": 4,
    "max_variables": 18,
}
gw_common = {
    "brute_force_threshold": 4,
    "rounds": 16,
    "seed": 7,
    "sdp_solver": "CLARABEL",
    "sdp_tolerance": 1e-7,
    "max_variables": 18,
}

qrr_r1 = 1m().solve(
    problem,
    {**qrr_common, "branching_rule": "r1"},
)
gw_r1 = GwBranchAndBoundQuboSolver().solve(
    problem,
    {**gw_common, "branching_rule": "r1"},
)

for result in (qrr_r1, gw_r1):
    validate_qubo_result(problem, result)
    assert result["status"] == "optimal"
    assert result["best_energy"] == oracle_energy
    assert result["metrics"]["tree_exhausted"] is True
    json.dumps(result, allow_nan=False)

print("QRR R1 nodes:", qrr_r1["metrics"]["nodes_explored"])
print("GW  R1 nodes:", gw_r1["metrics"]["nodes_explored"])

## 4. R2：row confidence

QRR 使用 selective composition；GW 使用原始 SDP matrix。

In [ ]:
qrr_r2 = QrbnbrQuboSolver().solve(
    problem,
    {
        **qrr_common,
        "branching_rule": "r2",
        "branching_matrix": "selective",
        "selective_rank": 3,
    },
)
gw_r2 = GwBranchAndBoundQuboSolver().solve(
    problem,
    {**gw_common, "branching_rule": "r2"},
)

for result in (qrr_r2, gw_r2):
    validate_qubo_result(problem, result)
    assert result["status"] == "optimal"
    assert result["best_energy"] == oracle_energy

print("QRR R2 selective nodes:", qrr_r2["metrics"]["nodes_explored"])
print("GW  R2 SDP nodes:", gw_r2["metrics"]["nodes_explored"])

## 5. 方法级摘要

下面只组织 solver 已返回的统计，不在 notebook 中重新实现任何算法。

In [ ]:
summary = []
for route, result, relaxation_count in (
    ("QRR / R1 correlation", qrr_r1, qrr_r1["metrics"]["qrr_subproblems"]),
    ("GW / R1 SDP", gw_r1, gw_r1["metrics"]["gw_subproblems"]),
    ("QRR / R2 selective", qrr_r2, qrr_r2["metrics"]["qrr_subproblems"]),
    ("GW / R2 SDP", gw_r2, gw_r2["metrics"]["gw_subproblems"]),
):
    summary.append({
        "route": route,
        "energy": result["best_energy"],
        "nodes": result["metrics"]["nodes_explored"],
        "pruned": result["metrics"]["nodes_pruned"],
        "relaxations": relaxation_count,
        "exact_leaves": result["metrics"]["exact_subproblems"],
    })

summary

## 结论与限制

- GW classical control 已与 QRR route 使用同一契约、问题和 tree certificate；
- 当前数值只验证一个确定性 S1 风格实例，不等同于论文统计图表复刻；
- 下一层实验应批量运行 S1 suite，记录 seeds、backend、节点分布和 wall time；
- BiqMac S2、optimized correction 与原论文具体随机样本仍未复现。